In [0]:
from pyspark.sql.functions import col, count, avg, round, sum

silver_table = "workspace.default.silver_accidents_hpms"
gold_table = "workspace.default.gold_accidents_hpms"

df = spark.table(silver_table)
print(f"Loaded {df.count()} rows from Silver")

In [0]:
# Gold Aggregate 1: Accidents by State + Severity + Weather
gold_state = df.groupBy("State", "Severity", "Weather_Condition", "Sunrise_Sunset") \
    .agg(
        count("*").alias("accident_count"),
        round(avg("Temperature_F"), 2).alias("avg_temp"),
        round(avg("Visibility_mi"), 2).alias("avg_visibility"),
        round(avg("Wind_Speed_mph"), 2).alias("avg_wind"),
        round(avg("SPEED_LIMIT"), 2).alias("avg_speed_limit"),
        round(avg("AADT"), 2).alias("avg_aadt"),
        round(avg("THROUGH_LANES"), 2).alias("avg_lanes")
    )

gold_state.write.format("delta").mode("overwrite").saveAsTable(gold_table)
print(f"Gold layer 1 done! Rows: {gold_state.count()}")
gold_state.show(5)

In [0]:
# Gold Aggregate 2: Insight - High speed limit roads vs accident severity
# Shows how HPMS speed limit data adds new insight
speed_severity = df.groupBy("SPEED_LIMIT", "Severity") \
    .agg(
        count("*").alias("accident_count"),
        round(avg("AADT"), 0).alias("avg_traffic_volume"),
        round(avg("THROUGH_LANES"), 1).alias("avg_lanes")
    ) \
    .orderBy("SPEED_LIMIT", "Severity")

speed_severity.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_speed_severity")
print("Gold layer 2 done - Speed limit vs Severity")
speed_severity.show(10)

In [0]:
# Gold Aggregate 3: Insight - Traffic volume (AADT) buckets vs accident severity
from pyspark.sql.functions import when

df_bucketed = df.withColumn("traffic_volume",
    when(col("AADT") < 5000, "Low (<5k)")
    .when(col("AADT") < 20000, "Medium (5k-20k)")
    .when(col("AADT") < 50000, "High (20k-50k)")
    .otherwise("Very High (>50k)")
)

aadt_severity = df_bucketed.groupBy("traffic_volume", "Severity") \
    .agg(
        count("*").alias("accident_count"),
        round(avg("Distance_mi"), 3).alias("avg_accident_distance")
    ) \
    .orderBy("traffic_volume", "Severity")

aadt_severity.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_traffic_severity")
print("Gold layer 3 done - Traffic volume vs Severity")
aadt_severity.show(10)

In [0]:
# Verify all gold tables
print("=== Top 10 States by Accident Count ===")
spark.sql("""
    SELECT State, SUM(accident_count) as total_accidents
    FROM workspace.default.gold_accidents_hpms
    GROUP BY State
    ORDER BY total_accidents DESC
    LIMIT 10
""").show()

print("=== Accidents by Speed Limit ===")
spark.sql("SELECT * FROM workspace.default.gold_speed_severity LIMIT 10").show()

print("=== Accidents by Traffic Volume ===")
spark.sql("SELECT * FROM workspace.default.gold_traffic_severity").show()